# Module 7 — EEGMMIDB(109) → BCI-IV-2a(9) ATCNet-3C Cross-Dataset

**Source:** all 109 EEGMMIDB subjects.  
**External target:** all 9 BCI-IV-2a subjects.  
**Input:** `(N, 22, 640)` at 160 Hz.  
**Classes:** `left`, `right`, `feet`.

This is deliberately separated from the 9-subject BCI LOSO benchmark. The final source model is trained on all 109 EEGMMIDB subjects. BCI labels are reserved for final scoring.

Two outputs are reported:
- **Strict:** BCI is not used for training, validation, normalization fitting, or checkpoint selection.
- **Transductive:** unlabeled BCI EEG is used only to estimate target median/IQR normalization; no BCI labels are used.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / DEVICE / SEED
# ============================================================
import os, gc, copy, time, random, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix, classification_report
warnings.filterwarnings("ignore")
SEED=42

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"]=str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic=True
        torch.backends.cudnn.benchmark=False
seed_everything()
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)

Device: cpu


In [2]:
# ============================================================
# CELL 2 — CACHE DISCOVERY
# ============================================================
PROJECT_ROOT=Path("/Users/ashokvarmabevara/Project2")
PROJECT_DIR=PROJECT_ROOT/"cross_dataset_mi_project"
CACHE_DIR=PROJECT_DIR/"cache"
RESULT_DIR=PROJECT_DIR/"results"/"module_7_cross_dataset"
RESULT_DIR.mkdir(parents=True,exist_ok=True)
CACHE_PATH=CACHE_DIR/"module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
if not CACHE_PATH.exists():
    candidates=sorted(CACHE_DIR.glob("*.h5"))
    preferred=[p for p in candidates if "160hz" in p.name.lower() or "module_5" in p.name.lower()]
    if not preferred: raise FileNotFoundError(f"No HDF5 cache in {CACHE_DIR}")
    CACHE_PATH=preferred[0]
print("Cache:",CACHE_PATH)
assert CACHE_PATH.exists()

Cache: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5


In [3]:
# ============================================================
# CELL 3 — METADATA / EXACT 109 + 9 SUBJECT CHECK
# ============================================================
def decode(v): return v.decode("utf-8") if isinstance(v,bytes) else str(v)
with h5py.File(CACHE_PATH,"r") as h5:
    X_shape=tuple(h5["X"].shape); X_dtype=str(h5["X"].dtype)
    meta={}
    for key in ["dataset","subject","run","recording_id","filename","absolute_path","harmonized_class"]:
        meta[key]=[decode(v) for v in h5["metadata"][key][:]]
cache_meta_df=pd.DataFrame(meta)
cache_meta_df.insert(0,"cache_index",np.arange(len(cache_meta_df),dtype=np.int64))
CLASSES=["left","right","feet"]; CLASS_TO_ID={c:i for i,c in enumerate(CLASSES)}; N_CLASSES=3
assert X_shape[1:]==(22,640); assert X_dtype=="float32"
EEGMMIDB_META=cache_meta_df[cache_meta_df["dataset"].astype(str).str.upper()=="EEGMMIDB"].copy()
BCI_META=cache_meta_df[cache_meta_df["dataset"].astype(str)=="BCI-IV-2a"].copy()
EEGMMIDB_META["subject"]=EEGMMIDB_META["subject"].astype(str); BCI_META["subject"]=BCI_META["subject"].astype(str)
SOURCE_SUBJECTS=sorted(EEGMMIDB_META["subject"].unique()); TARGET_SUBJECTS=sorted(BCI_META["subject"].unique())
print("Cache shape:",X_shape)
print("EEGMMIDB subjects:",len(SOURCE_SUBJECTS))
print("BCI-IV-2a subjects:",len(TARGET_SUBJECTS))
print("Total subjects:",len(SOURCE_SUBJECTS)+len(TARGET_SUBJECTS))
assert len(SOURCE_SUBJECTS)==109; assert len(TARGET_SUBJECTS)==9
print("PASS: 109 source + 9 external target = 118 subjects")

Cache shape: (9316, 22, 640)
EEGMMIDB subjects: 109
BCI-IV-2a subjects: 9
Total subjects: 118
PASS: 109 source + 9 external target = 118 subjects


In [4]:
# ============================================================
# CELL 4 — HDF5 LOADER + COVERAGE
# ============================================================
def load_indices(indices):
    indices=np.asarray(indices,dtype=np.int64)
    with h5py.File(CACHE_PATH,"r") as h5:
        return np.asarray(h5["X"][indices],dtype=np.float32)
print("EEGMMIDB epochs:",len(EEGMMIDB_META))
print("BCI epochs:",len(BCI_META))
print("Source class counts:")
display(EEGMMIDB_META["harmonized_class"].value_counts().reindex(CLASSES,fill_value=0).to_frame("epochs"))
print("Target class counts:")
display(BCI_META["harmonized_class"].value_counts().reindex(CLASSES,fill_value=0).to_frame("epochs"))

EEGMMIDB epochs: 7372
BCI epochs: 1944
Source class counts:


,epochs
harmonized_class,
left,2479
right,2438
feet,2455


Target class counts:


,epochs
harmonized_class,
left,648
right,648
feet,648


In [5]:
# ============================================================
# CELL 5 — NORMALIZATION
# ============================================================
class RobustChannelNormalizer:
    def __init__(self,eps=1e-6): self.eps=eps; self.center_=None; self.scale_=None
    def fit(self,X):
        X=np.asarray(X,np.float32); V=X.transpose(1,0,2).reshape(X.shape[1],-1)
        self.center_=np.median(V,axis=1)
        self.scale_=np.maximum(np.percentile(V,75,axis=1)-np.percentile(V,25,axis=1),self.eps)
        return self
    def transform(self,X):
        Z=(np.asarray(X,np.float32)-self.center_[None,:,None])/(self.scale_[None,:,None]+self.eps)
        return np.nan_to_num(Z,nan=0.0,posinf=0.0,neginf=0.0).astype(np.float32)

In [6]:
# ============================================================
# CELL 6 — ATCNET BLOCKS
# ============================================================
class CausalConv1d(nn.Module):
    def __init__(self,in_ch,out_ch,kernel_size,dilation=1):
        super().__init__(); self.pad=(kernel_size-1)*dilation; self.conv=nn.Conv1d(in_ch,out_ch,kernel_size,padding=self.pad,dilation=dilation,bias=False)
    def forward(self,x):
        y=self.conv(x); return y[...,:-self.pad] if self.pad else y

class TCNResidualBlock(nn.Module):
    def __init__(self,dim,filters=32,depth=2,kernel_size=4,dropout=0.30):
        super().__init__(); self.proj=nn.Conv1d(dim,filters,1) if dim!=filters else nn.Identity(); self.blocks=nn.ModuleList()
        for i in range(depth):
            d=2**i
            self.blocks.append(nn.ModuleDict({"c1":CausalConv1d(filters,filters,kernel_size,d),"b1":nn.BatchNorm1d(filters),"c2":CausalConv1d(filters,filters,kernel_size,d),"b2":nn.BatchNorm1d(filters),"drop":nn.Dropout(dropout)}))
    def forward(self,x):
        z=x.transpose(1,2); r=self.proj(z)
        for b in self.blocks:
            h=F.elu(b["b1"](b["c1"](r))); h=b["drop"](h); h=F.elu(b["b2"](b["c2"](h))); h=b["drop"](h); r=F.elu(r+h)
        return r.transpose(1,2)

class ATCNetConvBlock(nn.Module):
    def __init__(self,n_channels=22,F1=16,D=2,dropout=0.30):
        super().__init__(); F2=F1*D
        self.t=nn.Conv2d(1,F1,(1,64),padding=(0,32),bias=False); self.b1=nn.BatchNorm2d(F1)
        self.s=nn.Conv2d(F1,F2,(n_channels,1),groups=F1,bias=False); self.b2=nn.BatchNorm2d(F2)
        self.p1=nn.AvgPool2d((1,8)); self.d1=nn.Dropout(dropout)
        self.r=nn.Conv2d(F2,F2,(1,16),padding=(0,8),bias=False); self.b3=nn.BatchNorm2d(F2)
        self.p2=nn.AvgPool2d((1,7)); self.d2=nn.Dropout(dropout)
    def forward(self,x):
        z=x.unsqueeze(1); z=F.elu(self.b1(self.t(z))); z=F.elu(self.b2(self.s(z))); z=self.d1(self.p1(z)); z=F.elu(self.b3(self.r(z))); z=self.d2(self.p2(z)); return z.squeeze(2).transpose(1,2)

In [7]:
# ============================================================
# CELL 7 — ATCNET-3C
# ============================================================
class ATCNet3C(nn.Module):
    def __init__(self,n_channels=22,n_samples=640,n_classes=3,n_windows=5,F1=16,D=2,heads=2,tcn_filters=32):
        super().__init__(); self.n_windows=n_windows; self.conv=ATCNetConvBlock(n_channels,F1,D); self.feature_dim=F1*D
        self.attn=nn.ModuleList([nn.MultiheadAttention(self.feature_dim,heads,dropout=0.30,batch_first=True) for _ in range(n_windows)])
        self.norm=nn.ModuleList([nn.LayerNorm(self.feature_dim) for _ in range(n_windows)])
        self.tcn=nn.ModuleList([TCNResidualBlock(self.feature_dim,tcn_filters,2,4,0.30) for _ in range(n_windows)])
        self.head=nn.ModuleList([nn.Sequential(nn.Linear(tcn_filters,64),nn.ELU(),nn.Dropout(0.25),nn.Linear(64,n_classes)) for _ in range(n_windows)])
        old=self.training; self.eval()
        with torch.no_grad(): self.seq_len=int(self.conv(torch.zeros(2,n_channels,n_samples)).shape[1])
        if old: self.train()
    def forward(self,x):
        z=self.conv(x); out=[]
        for i in range(self.n_windows):
            w=z[:,i:self.seq_len-self.n_windows+i+1,:]; a,_=self.attn[i](w,w,w,need_weights=False); w=self.norm[i](w+a); w=self.tcn[i](w); out.append(self.head[i](w[:,-1,:]))
        return torch.stack(out,dim=0).mean(0)

m=ATCNet3C().to(device)
with torch.no_grad(): o=m(torch.randn(2,22,640,device=device))
print("Compressed sequence:",m.seq_len); print("Output:",tuple(o.shape))
assert tuple(o.shape)==(2,3)
del m; gc.collect()
print("PASS: ATCNet-3C forward")

Compressed sequence: 11
Output: (2, 3)
PASS: ATCNet-3C forward


In [8]:
# ============================================================
# CELL 8 — LOADER + AUGMENTATION + PREDICTION
# ============================================================
def augment_source(x):
    x=x.clone(); B=x.shape[0]
    if torch.rand(1,device=x.device).item()<0.35: x=x*torch.empty(B,1,1,device=x.device).uniform_(0.94,1.06)
    if torch.rand(1,device=x.device).item()<0.20: x=x+0.003*torch.randn_like(x)
    return x

def make_train_loader(X,y,batch_size=64):
    ds=TensorDataset(torch.from_numpy(np.asarray(X,np.float32)),torch.from_numpy(np.asarray(y,np.int64)))
    counts=np.bincount(y,minlength=3).astype(np.float64); inv=np.zeros(3); good=counts>0; inv[good]=1.0/counts[good]
    sampler=WeightedRandomSampler(torch.as_tensor(inv[y],dtype=torch.double),len(y),replacement=True)
    return DataLoader(ds,batch_size=batch_size,sampler=sampler,num_workers=0)

def make_eval_loader(X,batch_size=128):
    ds=TensorDataset(torch.from_numpy(np.asarray(X,np.float32)),torch.zeros(len(X),dtype=torch.long))
    return DataLoader(ds,batch_size=batch_size,shuffle=False,num_workers=0)

@torch.no_grad()
def predict_probs(model,X):
    model.eval(); out=[]
    for xb,_ in make_eval_loader(X): out.append(model(xb.to(device)).cpu().numpy())
    logits=np.concatenate(out,axis=0); logits-=logits.max(axis=1,keepdims=True); P=np.exp(logits); P/=P.sum(axis=1,keepdims=True)+1e-12
    return P.astype(np.float32)

In [9]:
# ============================================================
# CELL 9 — GROUPED SOURCE CV TO CHOOSE TRAINING LENGTH
# ============================================================
train_subj,val_subj=(lambda s: (s[20:],s[:20]))(sorted(SOURCE_SUBJECTS))
# Deterministic source-only subject split: first 20 validation, remaining 89 train.
# No BCI data is touched.
def arrays(meta_df,subjects):
    mask=meta_df["subject"].astype(str).isin(set(subjects)); idx=meta_df.loc[mask,"cache_index"].to_numpy(np.int64)
    X=load_indices(idx); y=meta_df.loc[mask,"harmonized_class"].map(CLASS_TO_ID).to_numpy(np.int64); return X,y
Xtr_raw,ytr=arrays(EEGMMIDB_META,train_subj); Xv_raw,yv=arrays(EEGMMIDB_META,val_subj)
norm=RobustChannelNormalizer().fit(Xtr_raw); Xtr=norm.transform(Xtr_raw); Xv=norm.transform(Xv_raw)

best_epochs=[]
for seed in (42,123):
    seed_everything(seed); model=ATCNet3C().to(device)
    counts=np.bincount(ytr,minlength=3).astype(np.float32); w=counts.sum()/(3*np.maximum(counts,1)); w/=w.mean()+1e-12
    criterion=nn.CrossEntropyLoss(weight=torch.tensor(w,dtype=torch.float32,device=device),label_smoothing=0.01)
    opt=optim.Adam(model.parameters(),lr=9e-4)
    sched=optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=0.90,patience=7,min_lr=1e-5)
    loader=make_train_loader(Xtr,ytr,64); best_loss=np.inf; best_epoch=1; wait=0
    for epoch in range(1,81):
        model.train()
        for xb,yb in loader:
            xb=augment_source(xb.to(device)); yb=yb.to(device); opt.zero_grad(set_to_none=True); loss=criterion(model(xb),yb)
            if torch.isfinite(loss): loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        P=predict_probs(model,Xv); vloss=-float(np.mean(np.log(np.clip(P[np.arange(len(yv)),yv],1e-8,1.0)))); sched.step(vloss)
        if vloss<best_loss-1e-5: best_loss=vloss; best_epoch=epoch; wait=0
        else: wait+=1
        if epoch==1 or epoch%10==0: print(f'seed {seed} epoch {epoch:03d} val_bAcc={balanced_accuracy_score(yv,P.argmax(1))*100:.2f}%')
        if wait>=18: break
    best_epochs.append(best_epoch); del model; gc.collect()

FINAL_EPOCHS=max(25,min(int(round(np.median(best_epochs))),80))
print('Chosen final epochs:',FINAL_EPOCHS)

seed 42 epoch 001 val_bAcc=41.10%
seed 42 epoch 010 val_bAcc=45.77%
seed 42 epoch 020 val_bAcc=47.69%
seed 123 epoch 001 val_bAcc=41.51%
seed 123 epoch 010 val_bAcc=49.30%
seed 123 epoch 020 val_bAcc=47.14%
Chosen final epochs: 25


In [10]:
# ============================================================
# CELL 10 — FINAL TRAINING ON ALL 109 EEGMMIDB SUBJECTS
# ============================================================
X_all_raw,y_all=arrays(EEGMMIDB_META,SOURCE_SUBJECTS)
X_bci_raw,y_bci=arrays(BCI_META,TARGET_SUBJECTS)

source_norm=RobustChannelNormalizer().fit(X_all_raw)
X_all=source_norm.transform(X_all_raw)
X_bci_strict=source_norm.transform(X_bci_raw)

# Target-only statistics. This uses target EEG values but NO labels.
target_norm=RobustChannelNormalizer().fit(X_bci_raw)
X_bci_transductive=target_norm.transform(X_bci_raw)

def train_full(seed):
    seed_everything(seed); model=ATCNet3C().to(device)
    counts=np.bincount(y_all,minlength=3).astype(np.float32); w=counts.sum()/(3*np.maximum(counts,1)); w/=w.mean()+1e-12
    criterion=nn.CrossEntropyLoss(weight=torch.tensor(w,dtype=torch.float32,device=device),label_smoothing=0.01)
    opt=optim.Adam(model.parameters(),lr=9e-4)
    sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=FINAL_EPOCHS,eta_min=2e-6)
    loader=make_train_loader(X_all,y_all,64)
    for epoch in range(1,FINAL_EPOCHS+1):
        model.train(); losses=[]
        for xb,yb in loader:
            xb=augment_source(xb.to(device)); yb=yb.to(device); opt.zero_grad(set_to_none=True); loss=criterion(model(xb),yb)
            if torch.isfinite(loss): loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step(); losses.append(float(loss.item()))
        sched.step()
        if epoch==1 or epoch%10==0 or epoch==FINAL_EPOCHS: print(f'final seed {seed} epoch {epoch:03d}/{FINAL_EPOCHS} loss={np.mean(losses):.4f}')
    return model

final_models=[train_full(seed) for seed in (42,123)]
print('PASS: final model trained on all 109 EEGMMIDB subjects')

final seed 42 epoch 001/25 loss=1.1021
final seed 42 epoch 010/25 loss=0.9412


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 11 — STRICT + TRANSDUCTIVE CROSS-DATASET RESULTS
# ============================================================
P_strict=np.mean(np.stack([predict_probs(m,X_bci_strict) for m in final_models],axis=0),axis=0)
P_trans=np.mean(np.stack([predict_probs(m,X_bci_transductive) for m in final_models],axis=0),axis=0)
pred_strict=P_strict.argmax(1); pred_trans=P_trans.argmax(1)

strict_acc=accuracy_score(y_bci,pred_strict)*100.0; strict_bacc=balanced_accuracy_score(y_bci,pred_strict)*100.0; strict_k=cohen_kappa_score(y_bci,pred_strict)
trans_acc=accuracy_score(y_bci,pred_trans)*100.0; trans_bacc=balanced_accuracy_score(y_bci,pred_trans)*100.0; trans_k=cohen_kappa_score(y_bci,pred_trans)

print('='*78); print('EEGMMIDB(109) → BCI-IV-2a(9)'); print('='*78)
print(f'STRICT        accuracy={strict_acc:.2f}%  bAcc={strict_bacc:.2f}%  kappa={strict_k:.4f}')
print(f'TRANSDUCTIVE  accuracy={trans_acc:.2f}%  bAcc={trans_bacc:.2f}%  kappa={trans_k:.4f}')
print(f'Target-normalization change: {trans_acc-strict_acc:+.2f} pp')

In [ ]:
# ============================================================
# CELL 12 — PER-SUBJECT RESULTS + SAVE
# ============================================================
rows=[]
for subject in TARGET_SUBJECTS:
    mask=BCI_META['subject'].astype(str).eq(str(subject)).to_numpy(); idx=np.flatnonzero(mask); yt=y_bci[idx]
    rows.append({
        'subject':subject,
        'strict_accuracy':accuracy_score(yt,pred_strict[idx])*100.0,
        'strict_bacc':balanced_accuracy_score(yt,pred_strict[idx])*100.0,
        'transductive_accuracy':accuracy_score(yt,pred_trans[idx])*100.0,
        'transductive_bacc':balanced_accuracy_score(yt,pred_trans[idx])*100.0,
    })
subject_results=pd.DataFrame(rows); subject_results['delta_pp']=subject_results['transductive_accuracy']-subject_results['strict_accuracy']
display(subject_results)
print('STRICT mean:',f"{subject_results.strict_accuracy.mean():.2f}%")
print('TRANSDUCTIVE mean:',f"{subject_results.transductive_accuracy.mean():.2f}%")
print('STRICT subjects >=70:',int((subject_results.strict_accuracy>=70).sum()),'/9')
print('TRANSDUCTIVE subjects >=70:',int((subject_results.transductive_accuracy>=70).sum()),'/9')

print('Strict confusion matrix:')
display(pd.DataFrame(confusion_matrix(y_bci,pred_strict,labels=[0,1,2],normalize='true'),index=CLASSES,columns=CLASSES).round(3))
print('Transductive confusion matrix:')
display(pd.DataFrame(confusion_matrix(y_bci,pred_trans,labels=[0,1,2],normalize='true'),index=CLASSES,columns=CLASSES).round(3))

csv_path=RESULT_DIR/'eegmmidb109_to_bci9_atcnet3c_results.csv'; subject_results.to_csv(csv_path,index=False)
protocol={'source':'EEGMMIDB','source_subjects':109,'target':'BCI-IV-2a','target_subjects':9,'input':[22,640],'sampling_rate_hz':160,'classes':CLASSES,'model':'ATCNet-3C','seeds':[42,123],'final_training_subjects':109,'strict_target_statistics':False,'transductive_target_statistics':True,'target_labels_used_for_training':False,'target_labels_used_for_selection':False}
protocol_path=RESULT_DIR/'eegmmidb109_to_bci9_protocol.json'; protocol_path.write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print('Saved:',csv_path); print('Saved:',protocol_path)